# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [3]:
print('datasize:', df.shape)
print('thông tin dữ liệu:')
df.info()

datasize: (1338, 7)
thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


## A.2. Missing values & Duplicate data

In [4]:
print('số lượng giá trị bị thiếu:\n', df.isnull().sum())
print('số lượng giá trị bị trùng:\n', df.duplicated().sum())

số lượng giá trị bị thiếu:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
số lượng giá trị bị trùng:
 1


## A.3. Invalid values

In [7]:
display(df.describe())

print("\nUnique values của 'sex':", df['sex'].unique())
print("Unique values của 'smoker':", df['smoker'].unique())
print("Unique values của 'region':", df['region'].unique())


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010



Unique values của 'sex': ['female' 'male']
Unique values của 'smoker': ['yes' 'no']
Unique values của 'region': ['southwest' 'southeast' 'northwest' 'northeast']


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [9]:
df['bmi_group'] = pd.cut(
    df['bmi'], 
    bins=[0, 25, 30, float('inf')], 
    labels=['Normal', 'Overweight', 'Obese'], 
    right=False 
)

print(df[['bmi', 'bmi_group']].head())

      bmi   bmi_group
0  27.900  Overweight
1  33.770       Obese
2  33.000       Obese
3  22.705      Normal
4  28.880  Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [ ]:
print("MEAN")
print(df[['age', 'bmi', 'children', 'charges']].mean())
print("\nMEDIAN")
print(df[['age', 'bmi', 'children', 'charges']].median())
print("\nMODE")
print(df.mode().iloc[0])


--- MEAN ---
age            39.207025
bmi            30.663397
children        1.094918
charges     13270.422265
dtype: float64

--- MEDIAN ---
age           39.000
bmi           30.400
children       1.000
charges     9382.033
dtype: float64

--- MODE ---
age                 18
sex               male
bmi               32.3
children             0
smoker              no
region       southeast
charges      1639.5631
bmi_group        Obese
Name: 0, dtype: object


## Group 2 — Dispersion

In [14]:
print("Độ lệch chuẩn")
print(df[['age', 'bmi', 'children', 'charges']].std())
print("\nMin & Max")
print(df[['age', 'bmi', 'children', 'charges']].agg(['min', 'max']))

Độ lệch chuẩn
age            14.049960
bmi             6.098187
children        1.205493
charges     12110.011237
dtype: float64

Min & Max
     age    bmi  children      charges
min   18  15.96         0   1121.87390
max   64  53.13         5  63770.42801


## Group 3 — Location and Shape

In [16]:
print("Độ lệch")
print(df[['age', 'bmi', 'children', 'charges']].skew())
print("\nĐộ nhọn")
print(df[['age', 'bmi', 'children', 'charges']].kurt())


Độ lệch
age         0.055673
bmi         0.284047
children    0.938380
charges     1.515880
dtype: float64

Độ nhọn
age        -1.245088
bmi        -0.050732
children    0.202454
charges     1.606299
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [18]:
chi_phi_hut_thuoc = df.groupby('smoker')['charges'].mean()
print("Chi phí trung bình theo tình trạng hút thuốc:")
print(chi_phi_hut_thuoc)
print(f"=> Chi phí người hút thuốc cao hơn {chi_phi_hut_thuoc['yes'] / chi_phi_hut_thuoc['no']:.2f} lần so với người không hút thuốc.")


Chi phí trung bình theo tình trạng hút thuốc:
smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64
=> Chi phí người hút thuốc cao hơn 3.80 lần so với người không hút thuốc.


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [23]:
cor_smoker = df[df['smoker']=='yes']['bmi'].corr(df[df['smoker']=='yes']['charges'])
cor_non_smoker = df[df['smoker']=='no']['bmi'].corr(df[df['smoker']=='no']['charges'])
print(f"Tương quan (Smoker): {cor_smoker:.2f}")
print(f"Tương quan (Non-smoker): {cor_non_smoker:.2f}")


Tương quan (Smoker): 0.81
Tương quan (Non-smoker): 0.08


Ở nhóm hút thuốc corr = 0.81 -> tương quan thuận, người hút thuốc mà BMI càng tăng thì phí bảo hiểm tăng
Ở nhóm không hút thuốc = 0.08 -> không tương quan, người không hút thuốc, việc tăng giảm bmi không làm thay đổi chi phí bảo hiểm

## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [21]:
print(df.groupby('region')['charges'].mean().sort_values(ascending=False))


region
southeast    14735.411438
northeast    13406.384516
northwest    12417.575374
southwest    12346.937377
Name: charges, dtype: float64


vùng southeast có phí bảo hiểm trung bình cao nhất

## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [22]:
print(df.groupby('children')['charges'].mean())

children
0    12365.975602
1    12731.171832
2    15073.563734
3    15355.318367
4    13850.656311
5     8786.035247
Name: charges, dtype: float64


Việc có con cái có làm tăng chi phí bảo hiểm so với người không có con (nhóm 0 con). Tuy nhiên, từ đứa con thứ 4 trở đi, chi phí trung bình lại có xu hướng giảm xuống.

## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [ ]:
cor_age_charges = df['age'].corr(df['charges'])
print(f"Hệ số tương quan giữa Tuổi và Chi phí: {cor_age_charges:.2f}")


Hệ số tương quan chung giữa Tuổi và Chi phí: 0.30


hệ số tương quan giữa tuổi và chi phí là 0.3 -> tương quan thuận nhưng ở mức trung bình yếu

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*

Tình trạng hút thuốc quyết định chi phí bảo hiểm, người hút thuốc phải chịu mức phi cao gấp 4 lần 
Tình trạng thừa cân (BMI cao) kết hợp với việc hút thuốc tạo rủi ro về sức khỏe dẫn đến chi phí bảo hiểm tăng
Tuổi tác là yếu tố cơ bản làm tăng chi phí đẩu đặn theo thời gian cho mọi khách hàng
khu vực sống và số lượng con cái có tác động yếu, không gây ra sự sai lệch chi phí quá lớn